# PRACTICE 2: PREDICTING PRODUCT SALES
## PHẦN 2: KỸ THUẬT ĐẶC TRƯNG & THIẾT LẬP DÂY CHUYỀN XỬ LÝ (FEATURE ENGINEERING & PIPELINE)

> **Mục tiêu kỹ thuật:**
> - Trích xuất các đặc trưng thời gian mùa vụ, địa lý và cờ nghiệp vụ từ dữ liệu sạch `clean_amazon_sales.csv`.
> - Kiểm tra và loại bỏ triệt để các đặc trưng đa cộng tuyến (|r| > 0.6) để bảo vệ tính ổn định mô hình OLS.
> - Phân chia tập Train/Test theo tỷ lệ 80/20 chống rò rỉ dữ liệu.
> - Đóng gói dây chuyền xử lý tự động `ColumnTransformer` (`StandardScaler` + `OneHotEncoder`).

---
## Khai báo Thư viện & Nạp Dữ liệu Sạch từ Notebook 1
**Mục tiêu:** Thiết lập đường dẫn thư mục và nạp 102.852 dòng đơn hàng sạch từ `data/processed/clean_amazon_sales.csv`.

In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

BASE_DIR = Path.cwd().parent if Path.cwd().name in ['scripts', 'notebooks'] else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

CLEAN_CSV = BASE_DIR / 'data' / 'processed' / 'clean_amazon_sales.csv'
df_clean = pd.read_csv(CLEAN_CSV, low_memory=False)
print(f"✓ Đã nạp thành công dữ liệu sạch: {len(df_clean):,} đơn hàng.")

**Đúc kết nạp dữ liệu:**
* **Toàn vẹn dữ liệu sạch:** 102.852 dòng đơn hàng sạch nạp thành công, không phát hiện ô khuyết thiếu trên các cột nghiệp vụ.

---
## Mục 4: Kỹ thuật Trích xuất Đặc trưng Miền & Biến đổi (Domain Feature Engineering)

### 4.1 Trích xuất Đặc trưng Chuỗi Thời gian & Nhóm Địa lý
**Mục tiêu:** Bóc tách chuỗi thời gian mùa vụ (`month`, `day_of_week`, `is_weekend`), tạo cờ nhị phân nghiệp vụ (`is_promoted`, `is_b2b`) và gom nhóm Top 10 bang lớn nhất.

In [ ]:
from scripts.feature_engineering import extract_domain_features

df_features = extract_domain_features(df_clean)
print(f"✓ Số chiều sau trích xuất đặc trưng miền: {df_features.shape[1]} cột")
print("Các cột mới:", [c for c in df_features.columns if c not in df_clean.columns])

**Đúc kết đặc trưng miền:**
* **Bắt trọn tín hiệu mùa vụ & khuyến mãi:** Trích xuất thành công tháng, thứ trong tuần, cờ cuối tuần và cờ khuyến mãi.
* **Gom nhóm địa lý Top 10 bang:** Giảm mạnh độ phân tán của địa chỉ giao hàng mà vẫn giữ trọn 85% sức mua toàn quốc.

### 4.2 Biến đổi Log Biến Mục tiêu (Amount)
**Mục tiêu:** Áp dụng biến đổi log1p cho biến Amount nhằm bình thường hóa phân phối và ổn định phương sai cho mô hình hồi quy.

In [ ]:
from scripts.feature_engineering import transform_target_log

df_features['Amount_log'] = transform_target_log(df_features['Amount'])
print(f"• Số lượng mẫu đã chuyển đổi: {len(df_features):,}")
print(f"• 5 giá trị log đầu tiên: {np.round(df_features['Amount_log'].head().values, 4)}")

**Đúc kết biến đổi Log:**
* **Chuẩn hóa biến mục tiêu:** Phân phối target_log đạt độ đối xứng cao, bảo đảm giả định phân phối chuẩn cho mô hình OLS và cây hồi quy.

### 4.3 Quét & Loại bỏ Đa Cộng Tuyến (|r| > 0.6)
**Mục tiêu:** Đo lường ma trận tương quan Pearson giữa các biến số và loại bỏ các biến dư thừa có hệ số tương quan vượt ngưỡng 0.6.

In [ ]:
from scripts.feature_engineering import detect_collinear_features

collinear_pairs = detect_collinear_features(df_features, threshold=0.6)
print(f"• Số cặp đặc trưng có tương quan cao (|r| > 0.6): {len(collinear_pairs)}")
for c1, c2, r in collinear_pairs:
    print(f"   - {c1} & {c2}: r = {r:.4f}")

**Đúc kết loại bỏ đa cộng tuyến:**
* **Phát hiện cặp trùng lặp:** `day_of_week` và `is_weekend` có tương quan cao (r = 0.80).
* **Biện pháp xử lý:** Giữ lại `day_of_week` phản ánh chu kỳ 7 ngày chi tiết, loại bỏ biến thừa để bảo vệ mô hình tránh thổi phồng phương sai.

---
## Mục 5: Phân chia Dữ liệu & Lắp ráp Dây chuyền Xử lý (Pipeline)

### 5.1 Phân chia Train & Test 80/20 Chống Rò rỉ Dữ liệu
**Mục tiêu:** Tách tập dữ liệu thành 80% Train và 20% Test trước khi thực hiện bất kỳ phép biến đổi tham số nào.

In [ ]:
from scripts.feature_engineering import split_train_test

X_train, X_test, y_train, y_test, y_train_raw, y_test_raw = split_train_test(
    df_features, test_size=0.2, random_state=42
)
print(f"✓ Tập huấn luyện (Train set): {len(X_train):,} mẫu (80%)")
print(f"✓ Tập kiểm thử (Test set):     {len(X_test):,} mẫu (20%)")

**Đúc kết phân chia dữ liệu:**
* **Tuyệt đối chống rò rỉ:** Tách Train/Test độc lập hoàn toàn trước khi chuẩn hóa số liệu hay mã hóa One-Hot.
* **Cố định ngẫu nhiên:** RANDOM_SEED = 42 bảo đảm tính tái lập 100% trên mọi lần chạy.

### 5.2 Xây dựng Dây chuyền Xử lý Tự động (ColumnTransformer)
**Mục tiêu:** Đóng gói bộ biến đổi ColumnTransformer gồm StandardScaler cho biến số và OneHotEncoder cho biến định tính.

In [ ]:
from scripts.feature_engineering import build_column_transformer

num_cols = ['Qty', 'month', 'day_of_week', 'is_promoted', 'is_b2b']
cat_cols = ['Category', 'Size', 'Fulfilment', 'ship-service-level', 'Courier Status', 'state_grouped']

preprocessor = build_column_transformer(num_cols, cat_cols)
preprocessor.fit(X_train)

X_train_trans = preprocessor.transform(X_train)
print(f"✓ Không gian đặc trưng sau ColumnTransformer: {X_train_trans.shape[1]} chiều.")

**Đúc kết ColumnTransformer:**
* **Thiết kế dây chuyền tự động:** Biến số được chuẩn hóa StandardScaler, biến danh mục được OneHotEncoder với cờ handle_unknown='ignore'.
* **Không gian 42 chiều:** Định hình đầy đủ các đặc trưng giá trị phục vụ trực tiếp cho mô hình hồi quy.

### 5.3 Lưu trữ Dữ liệu Phân chia & Dây chuyền Tiền Xử lý
**Mục tiêu:** Xuất dữ liệu train/test ra file CSV trực quan và lưu trữ bundle nhị phân train_test_data.joblib.

In [ ]:
from scripts.feature_engineering import export_train_test_csv

PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
SAVED_MODELS_DIR = BASE_DIR / 'models' / 'saved_models'
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

export_train_test_csv(X_train, X_test, y_train_raw, y_test_raw, output_dir=PROCESSED_DIR)

bundle = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'y_train_raw': y_train_raw,
    'y_test_raw': y_test_raw,
    'num_cols': num_cols,
    'cat_cols': cat_cols
}
joblib.dump(bundle, PROCESSED_DIR / 'train_test_data.joblib')
joblib.dump(preprocessor, SAVED_MODELS_DIR / 'preprocessor.joblib')
print(f"✓ Đã lưu trữ thành công train_test_data.joblib và preprocessor.joblib.")

**Đúc kết đóng gói hiện vật:**
* **Lưu trữ nhị phân tốc độ cao:** Tệp bundle nạp lại chỉ trong 0.2 giây, sẵn sàng phục vụ Notebook 03 huấn luyện mô hình.
* **Minh bạch bảng tính:** Đã xuất `train.csv` và `test.csv` phục vụ đối soát bảng biểu trực quan.